# Context Adaptive Dictionary Analysis (CADA) — Example Pipeline

This notebook walks through the four-stage CADA pipeline on a minimal
**general morality** example. It is intended as a self-contained minimal illustration
of the method; it does **not** reproduce the paper's results.

**Prerequisites** — run once before opening this notebook:

```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

Per-decade word embeddings are also required. The repository ships a
vocabulary-pruned sample covering **1850**, **1900**, and **1950** from the Google Books
HistWords release ([Hamilton et al., 2018](https://doi.org/10.48550/arXiv.1605.09096))
in `aux/sample_embeddings/` for illustration purposes. These three slices are sufficient to run all four
steps below. To run the full pipeline across 1800–1990, download the complete
embedding set from <https://nlp.stanford.edu/projects/histwords/> and place
the files in `aux/embeddings/` — the scripts will detect the available decades
automatically.

**Note:** Because the embeddings in this repository are based on an (unsystematic) sample of total vocabulary 
they are **not** suitable for actual analysis. The following is merely an illustration of the pipeline.

In [1]:
from pathlib import Path
import shutil
import pandas as pd

from kernel import get_kernel_dictionary
from expansion import expand_kernel_lgde
from relevance import compute_moral_relevance
from scoring import score_docs

HERE = Path(".").resolve()
WORK = HERE / "example_output"
WORK.mkdir(exist_ok=True)

# Point at the bundled sample embeddings (1850, 1900, 1950).
# To use the full historical range, change this to HERE / "aux" / "embeddings"
# after placing the full HistWords files there — no other changes required.
SAMPLE_EMB = HERE / "aux" / "sample_embeddings";

## Step 0 — Seed Dictionary

The pipeline starts from a manually curated seed dictionary: a set of
words codifying expert derived semantic Instantiations of a target construct.
Here we define a simple **general morality** toy example.

Specifically, the list below consists of a small subset of "Morality-General" words taken from the original 
[Moral Foundations Dictionary (Graham & Haidt, 2012)](https://moralfoundations.org/other-materials/). 
These specific words capture generalized moral language rather than mapping to a single, distinct Moral Foundation.


The pipeline begins with a manually curated seed dictionary: a set of words 
representing expert-derived semantic instantiations of a target construct. 

> **To load seeds from a file instead**, replace the cell below with:
> ```python
> # Expected CSV format — two columns, no index:
> #   foundation,word
> #   moral.virtue,good
> #   moral.vice,evil
> #   ...
> seeds = pd.read_csv("seeds.csv")
> ```

In [2]:
# Examplary general-morality seed dictionary from MFD1
# 'foundation' labels the sub-construct; 'word' is the seed term.
seeds =pd.DataFrame([
    {"foundation": "MoralityGeneral", "word": "evil"},
    {"foundation": "MoralityGeneral", "word": "bad"},
    {"foundation": "MoralityGeneral", "word": "wrong"},
    {"foundation": "MoralityGeneral", "word": "offensive"},
    {"foundation": "MoralityGeneral", "word": "transgression"},
    {"foundation": "MoralityGeneral", "word": "moral"},
    {"foundation": "MoralityGeneral", "word": "good"},
    {"foundation": "MoralityGeneral", "word": "correct"},
    {"foundation": "MoralityGeneral", "word": "ethical"},
    {"foundation": "MoralityGeneral", "word": "upstanding"}
])

## Save seeds 
seeds_file = WORK / "seeds.csv"
seeds.to_csv(seeds_file, index=False)

print(seeds.groupby("foundation")["word"].apply(list).to_string())

foundation
MoralityGeneral    [evil, bad, wrong, offensive, transgression, m...


**Note:** In real applications a set of words this small is likely almost always insufficient to capture a construct at sufficient granularity. This example is purely illustrative.

## Step 1 — Kernel Construction (`kernel.py`)

`get_kernel_dictionary` filters the seed list to a *diachronically stable*
kernel by applying two criteria across all embedding decades:

1. **Frequency filter** — each word must reach a minimum Google Ngram
   frequency (`freq_cutoff`) in every decade considered.
2. **Semantic-change filter** — cumulative and decadal cosine drift must
   stay below a percentile threshold of a null distribution derived from
   construct-neutral reference words (`stopword_change.csv`).

The script automatically detects which decades are available in
`embedding_folder`, so no year range needs to be specified explicitly. 
`reference_file` can be set to point to a .csv file containing a set of words assumed
 to be semantically stable against which change is evaluated. If none is supplied 
 the function uses a stopword list.


In [3]:
%%capture
kernel_file = WORK / "kernel.csv"

get_kernel_dictionary(
     input_file= seeds_file,
     output_file= kernel_file,
     embedding_folder=HERE / "aux" / "sample_embeddings",
     freq_cutoff=1e-6,
     change_percentile=0.95,
     
 );

We can now look at which words were filtered due to semantic instability (or infrequency):

In [4]:
# Print deleted words for inspection.
kernel = pd.read_csv(kernel_file)
dropped_words = symmetric_diff = pd.concat([seeds, kernel]).drop_duplicates(keep=False)
print(f"Dropped {len(dropped_words)} words from a total of {len(seeds)} seeds:")
print(dropped_words.groupby("foundation")["word"].apply(list).to_string())

print("Kernel dictionary after filtering:")
print(kernel.groupby("foundation")["word"].apply(list).to_string())

Dropped 4 words from a total of 10 seeds:
foundation
MoralityGeneral    [offensive, transgression, ethical, upstanding]
Kernel dictionary after filtering:
foundation
MoralityGeneral    [evil, bad, wrong, moral, good, correct]


Four of the ten seed terms were removed by the kernel filter.

-  *Offensive* and *transgression* (Semantic Shift): Offensive shifted from an honor-culture and military register toward a psychological focus on subjective discomfort. Transgression evolved from a neutral context of physical boundary-crossing to an increasingly moralized sense (first only in theological and later also in secular discourse).
- *upstanding* (Both Criteria): This term had very low frequency in earlier decades and shifted over time from a literal physical posture to a figurative moral trait.
- *ethical* (Low Frequency): This exclusion reflects low usage frequency in early decades rather than semantic drift. The word originated in specialized philosophical Latin before gradually entering colloquial usage over the 19th century.

Filtering the kernel for usage-frequency safeguards your model. Sparse embeddings yield unreliable representations that can potentially bias the expansion phase (see below). If you want to disable this feature and skip filtering the kernel dictionary by historical usage frequency, set: `fetch_ngrams=False`.


## Step 2 — Dictionary Expansion (`expansion.py`)

The kernel is expanded into a larger dictionary by finding semantically
related words in each decade's embedding space. Three methods are available:

| Method | Description | Key parameters |
|---|---|---|
| `severability` (default) | Graph-based; detects local community structure around each seed in a continuous k-NN graph via the severability criterion | `k`, `t` |
| `siwo` | Alternative graph-based detector on the same CkNN graph | `k` |
| `cosine` | Graph-free; keeps every vocabulary word whose cosine similarity to any seed exceeds a threshold | `cosine_threshold` |

We use `cosine` here for speed. The graph-based methods are more principled and often less noisy
but computationally heavier (especially `severability` on large vocabularies).
Available embedding decades are detected automatically from `embedding_folder`.

In [5]:
expanded_file = WORK / "expanded.csv"

expanded_dic = expand_kernel_lgde(
    kernel_file=kernel_file,
    output_file=expanded_file,
    method="cosine",
    cosine_threshold=0.3,
    embedding_folder=SAMPLE_EMB,
    verbose=False
)

expanded = pd.read_csv(expanded_file)
print(f"Expanded dictionary: {len(expanded)} words total")
print(expanded.groupby("foundation")["word"].count().to_string())



Expanded dictionary: 580 words total
foundation
MoralityGeneral    580


## Step 3 — Moral-Relevance Trajectories (`relevance.py`)

For each (foundation, word) pair, `compute_moral_relevance` computes a
per-decade *moral-relevance score*: the soft-max share of the word's
proximity to its foundation centroid relative to a set of neutral reference
subspaces (`neutral.world`, `neutral.mind`, `neutral.society`). The result
is a column `p{YYYY}` for each scored decade.

The available embedding decades and the interpolation range are both inferred
automatically from `embedding_folder`, so the output never extrapolates
beyond the supplied data.

Note that with three sample decades LOESS smoothing is technically possible but
produces an under-identified fit. Setting `smooth_interpolate=False` in applied (non-illustrative)
context where this occurs may be advisable.

In [6]:
scored_dict_file = WORK / "scored_dictionary.csv"

compute_moral_relevance(
    dictionary_file=expanded_file,  # the expanded dictionary from the previous step
    output_file=scored_dict_file,    
    kernel_file=kernel_file,        # kernel centroid reference
    temp=0.05,                      # lower temp value = more discriminative output; usually best to iteratively calibrate
    smooth_interpolate=True,        # Set to False to only return scores for embedding years
    embedding_folder=SAMPLE_EMB,
    verbose=False
)

scored_dict = pd.read_csv(scored_dict_file)
print(f"Scored dictionary: {len(scored_dict)} words")
scored_dict.sort_values('p1850', ascending=False).head(10)

Scored dictionary: 580 words


,foundation,word,p1850,p1851,p1852,p1853,p1854,p1855,p1856,p1857,...,p1941,p1942,p1943,p1944,p1945,p1946,p1947,p1948,p1949,p1950
2,MoralityGeneral,evil,0.993846,0.993846,0.993846,0.993846,0.993846,0.993846,0.993660,0.993474,...,0.986373,0.986424,0.986474,0.986525,0.986575,0.986626,0.986677,0.986727,0.986778,0.986829
4,MoralityGeneral,bad,0.978436,0.978436,0.978436,0.978436,0.978436,0.978436,0.978587,0.978737,...,0.991718,0.991877,0.992037,0.992196,0.992356,0.992516,0.992675,0.992835,0.992995,0.993154
5,MoralityGeneral,wrong,0.977704,0.977704,0.977704,0.977704,0.977704,0.977704,0.977820,0.977936,...,0.987295,0.987400,0.987506,0.987611,0.987717,0.987823,0.987928,0.988034,0.988139,0.988245
3,MoralityGeneral,good,0.976037,0.976037,0.976037,0.976037,0.976037,0.976037,0.975966,0.975896,...,0.980160,0.980373,0.980586,0.980799,0.981012,0.981224,0.981437,0.981650,0.981863,0.982076
1,MoralityGeneral,moral,0.891076,0.891076,0.891076,0.891076,0.891076,0.891076,0.889915,0.888754,...,0.845588,0.845936,0.846285,0.846634,0.846982,0.847331,0.847680,0.848028,0.848377,0.848726
255,MoralityGeneral,morally,0.829171,0.829171,0.829171,0.829171,0.829171,0.829171,0.826041,0.822910,...,0.752498,0.754717,0.756935,0.759154,0.761372,0.763590,0.765809,0.768027,0.770245,0.772464
0,MoralityGeneral,correct,0.735240,0.735240,0.735240,0.735240,0.735240,0.735240,0.733877,0.732515,...,0.771815,0.774723,0.777631,0.780539,0.783447,0.786355,0.789263,0.792171,0.795079,0.797987
209,MoralityGeneral,unthankful,0.708897,0.708897,0.708897,0.708897,0.708897,0.708897,0.701329,0.693760,...,0.092702,0.086097,0.079492,0.072888,0.066283,0.059678,0.053074,0.046469,0.039864,0.033260
188,MoralityGeneral,evils,0.660711,0.660711,0.660711,0.660711,0.660711,0.660711,0.657880,0.655048,...,0.465831,0.464350,0.462870,0.461389,0.459909,0.458428,0.456947,0.455467,0.453986,0.452506
200,MoralityGeneral,doer,0.621634,0.621634,0.621634,0.621634,0.621634,0.621634,0.623234,0.624834,...,0.611070,0.608554,0.606038,0.603523,0.601007,0.598491,0.595975,0.593460,0.590944,0.588428


## Step 4 — Score Documents (`scoring.py`)

`score_docs` applies the scored dictionary to a corpus. The input DataFrame
must have a `text` column and a `year` column. For each document the function
returns one score per foundation plus a `moral_nonmoral_ratio`.

The `aggregation` argument controls normalisation:

- **`per_moral`** (default) — divides by the number of dictionary-matched
  tokens, expressing each foundation as its *share of the document's moral
  signal*.
- **`per_total`** — divides by total content tokens; closer to a LIWC-style
  rate.

If a document's year falls between scored decades, the closest available
year is used (with a warning).

Below is a minimal synthetic example corpus consisting of six sentences from three rough historical periods:

In [7]:
# Copy the scored dictionary into dictionaries/ so score_docs can locate it.
dest = HERE / "dictionaries" / "scored_dictionary.csv"
shutil.copy(scored_dict_file, dest)


docs = pd.DataFrame({
    "text": [
        # ── moral: acting with moral responsibility ────────────
        "He was guided by the notion that a man's worth lay wholly in his honest "
        "and principled dealings with others — for nothing, he held, was more "
        "unthankful than to receive another's trust and betray it.",

        "She was not indifferent to the harm and suffering she saw around her, and her whole "
        "upright and honest life had been directed toward its relief.",

        "It would have been foolish and indifferent to look away — the suffering "
        "was plain, and to ignore it was to be complicit in it.",

        # ── non-moral: going through some practical, unremarkable task ──────
        "It was a foolish, indifferent sort of errand — neither urgent nor easy "
        "to put off, the kind of thing that fills a morning without obvious purpose.",

        "She had a – perhaps unjustified – notion of how long the journey would take, though the roads "
        "were unfamiliar and the maps not entirely to be trusted.",

        "He worked from a principled set of legal documents with only a rough notion "
        "of the final tally — an unthankful task, but he filed the report "
        "safely before the deadline."
    ],
    "year": [1851, 1903, 1949, 1857, 1900, 1945],
})

You can see that vocabulary overlaps substantially between the two moral and non-moral sets of documents across eras. 
Despite this overlap, a CADA should of course still be able to differentiate between their connotation based on the context of their historical usage:

In [8]:
scored_docs = score_docs(docs, 
                         dic_type="scored_dictionary",    # points to the scored dictionary we just created
                         aggregation="per_total")         # Use aggregation="per_moral" for score/moralwords instead of score/totalwords
scored_docs[["text", "year", "MoralityGeneral"]]

Processed: 0   0% |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
Processed: 5  83% |⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱    | Elapsed Time: 0:00:00 ETA:   0:00:00
Processed: 6 100% |⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱⏱| Elapsed Time: 0:00:00 Time:  0:00:00


,text,year,MoralityGeneral
0,He was guided by the notion that a man's worth...,1851,0.117359
1,She was not indifferent to the harm and suffer...,1903,0.133354
2,It would have been foolish and indifferent to ...,1949,0.128628
3,"It was a foolish, indifferent sort of errand —...",1857,0.052270
4,She had a – perhaps unjustified – notion of ho...,1900,0.028219
5,He worked from a principled set of legal docum...,1945,0.020419


Indeed, it is immediately obvious that CADA achieves perfect separation between moral and non-moral content on this (admittedly highly constructed) example. 


The table below compares three scoring approaches: MFD1.0 (which can be though of as an expert expansion of the seed dictionary in Step 0, scored via standard LIWC-style word counting), the unweighted CADA  (a computationally derived diachronic expansion of that same seed, likewise scored by word count), and CADA  (the full method, which additionally weights each word's contribution by its historically estimated moral relevance in the document's period). Note that CADA scores are not directly comparable in absolute magnitude to the other two, as they reflect probability-weighted rather than binary token counts. 

| Doc | Year | Content | MFD1 | CADA (unweighted) | CADA |
|-----|------|-------|-----:|------------------:|-----:|
| 0 | 1851 | moral | 0.231 | 0.308 | 0.117 |
| 1 | 1903 | moral | 0.250 | 0.375 | 0.133 |
| 2 | 1949 | moral | 0.000 | 0.333 | 0.129 |
| 3 | 1857 | non-moral | 0.000 | 0.300 | 0.052 |
| 4 | 1900 | non-moral | 0.000 | 0.143 | 0.028 |
| 5 | 1945 | non-moral | 0.143 | 0.286 | 0.020 |
| -|  - | - |- |- | - |
| | | **Moral/Non-Moral Ratio** | 3.4× | 1.4× | 3.7× |
| | | **AUC** | 0.78 | 1.00 | 1.00 |


**Note:** As elaborated in the main paper, the results of the method may be unstable over short texts. Inference over sentence-level analysis (at the disaggregated level) is not generally recommended.